# Nexus · Studio portretów
Wgraj zdjęcie jednej osoby → 10 osobnych zdjęć z różnymi emocjami i pozami.

Wybierz **Środowisko wykonawcze → Zmień typ środowiska → GPU**. Model jest duży: potrzebuje dużo RAM i miejsca na dysku, nawet z offloadem. Pełny profil preferuje jakość; `4bit` zmniejsza pamięć. Dopasowanie do GPU oraz jakość trzeba sprawdzić w Twojej sesji. Nie uruchamiaj równolegle generatora wideo na tym samym GPU.

Panel tworzy zewnętrzny, tymczasowy link Gradio chroniony hasłem. Zdjęcia przetwarza Twoje środowisko Colab; pobierz ZIP przed wyłączeniem sesji. GPU może być płatne w zależności od planu. Nie jest wymagany klucz OpenAI.

In [ ]:
import torch
import shutil
assert torch.cuda.is_available(), "Włącz GPU w ustawieniach środowiska Colab."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
print("Wolny dysk:", round(shutil.disk_usage("/content").free / 2**30, 1), "GB")
print("Pobranie pełnych wag wymaga dziesiątek GB dysku; sprawdź zasoby przed kontynuacją.")


## 1. Pobierz kod i zależności
Pierwsze uruchomienie wymaga internetu. Notebook klonuje osobny katalog, nie zmienia Twoich innych projektów. Nie reinstaluje ręcznie PyTorcha.

In [ ]:
import os, sys, subprocess, tempfile
from pathlib import Path
repo_dir = Path(tempfile.mkdtemp(prefix="nexus-portrait-")) / "nexus-ai"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "feat/portrait-studio",
                "https://github.com/Damianwojownik/nexus-ai.git", str(repo_dir)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(repo_dir / "services/portrait_studio/requirements.txt")], check=True)
sys.path.insert(0, str(repo_dir))
print("Kod studia gotowy.")


## 2. Wybierz profil pamięci
`full` używa BF16/FP16, `4bit` kwantyzuje transformer i enkoder tekstu. `sequential` oszczędza VRAM i jest wolniejszy. Nie ma gwarancji, że model zmieści się w każdej sesji Colaba. Po zmianie ustawień zamknij panel i uruchom środowisko ponownie.

Jeśli działał wcześniej inny model, zrestartuj środowisko, aby zwolnić pamięć GPU.

In [ ]:
os.environ["NEXUS_PORTRAIT_PROFILE"] = "4bit"  # "full" dla większego RAM/GPU
os.environ["NEXUS_PORTRAIT_OFFLOAD"] = "sequential"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"


## 3. Uruchom panel
Ustaw własne hasło (minimum 12 znaków). Login to **nexus**. Hasło nie jest wyświetlane ani zapisywane w notebooku. Pierwsze naciśnięcie „Wygeneruj 10 zdjęć” pobierze i załaduje model.

Najlepszy materiał wejściowy: ostre zdjęcie w dobrym świetle, bez filtra, jedna osoba. Do póz rąk preferuj sylwetkę zamiast samej twarzy. Po ukończeniu powiększ zdjęcia, sprawdź rysy twarzy, oczy i palce; ponów nietrafione ujęcia.

In [ ]:
from getpass import getpass
from services.portrait_studio.app import build_app
password = getpass("Hasło do studia (minimum 12 znaków): ")
assert len(password) >= 12, "Użyj hasła o długości co najmniej 12 znaków."
studio = build_app()
studio.launch(share=True, auth=("nexus", password), server_name="127.0.0.1",
              max_file_size="20mb", show_error=False, debug=True)


## Wyniki i ograniczenia
Pobierz ZIP z panelu. Zawiera gotowe PNG i manifest z parametrami; oryginalne referencje nie są dołączane. Przy zatrzymaniu lub błędzie zachowany jest częściowy zestaw. Ponowienie jednego ujęcia nie zmienia pozostałych.

To generator oparty na modelu edycji zdjęć, nie gwarancja identyczności ani „najlepszej jakości na świecie”. Ta wersja wymaga testu rzeczywistego renderowania na GPU. Pełna instrukcja: `services/portrait_studio/README.md`.